[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [SQLAlchemy, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlalchemy-deep-dive.html)

# Reading Results &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell builds `scratch/college.db` as the notebook's Setup did, and makes what its worked
examples made: the engine, the queries `TRANSCRIPT`, `BY_EMAIL`, `BY_PROGRAM` and `CREDITS_EARNED`,
and `transcript`. Run it first. The tasks do not depend on one another, and the last cell removes the
scratch folder.


In [1]:
import json
import shutil
import sqlite3
from pathlib import Path

import sqlalchemy
from sqlalchemy import create_engine, event, text
from sqlalchemy.pool import StaticPool

SCRATCH = Path("scratch")
shutil.rmtree(SCRATCH, ignore_errors=True)
SCRATCH.mkdir()
DATABASE = SCRATCH / "college.db"

NAMES = [
    "Ana Reyes", "Ben Okafor", "Chloe Martin", "Daniel Kim", "Elena Petrova", "Felix Wagner",
    "Grace Lin", "Hassan Ali", "Isabel Costa", "Jonas Berg", "Keiko Tanaka", "Liam Murphy",
    "Maya Patel", "Noah Andersen", "Olivia Brandt", "Pavel Novak", "Quinn Harper", "Rosa Delgado",
    "Sam Ito", "Tara Nilsen", "Umar Farouk", "Vera Kowalski", "Wes Carter", "Yara Haddad",
    "Aoife O'Brien",
]
PROGRAMS = ["Biology", "Computer Science", "Mathematics", "Psychology", "History"]
TERMS = [("Fall 2024", "2024-08-26"), ("Spring 2025", "2025-01-13"), ("Fall 2025", "2025-08-25"),
         ("Spring 2026", "2026-01-12")]
STUDENTS = [(name, f"{name[0]}{name.split()[-1]}@college.edu".lower().replace("'", ""),
             PROGRAMS[i % len(PROGRAMS)], TERMS[i % 3][1]) for i, name in enumerate(NAMES)]
COURSES = [
    ("BIO-101", "Introduction to Biology", "Biology", 4),
    ("CHE-110", "General Chemistry", "Chemistry", 4),
    ("MAT-120", "Calculus I", "Mathematics", 4),
    ("MAT-121", "Calculus II", "Mathematics", 4),
    ("CSC-101", "Programming I", "Computer Science", 3),
    ("CSC-201", "Data Structures", "Computer Science", 3),
    ("ENG-105", "Composition", "English", 3),
    ("HIS-110", "World History", "History", 3),
    ("PSY-101", "Introduction to Psychology", "Psychology", 3),
    ("STA-200", "Statistics", "Mathematics", 3),
]
GRADES = ["A", "A-", "B+", "B", "B-", "C+", "C", "C-", "D", "F"]

# One section of every course in every term, so the section of course c in term t has id (t - 1) * 10 + c.
SECTIONS = [(course, term, 30) for term in range(1, len(TERMS) + 1) for course in range(1, len(COURSES) + 1)]

# Three courses a term for every student, from the term they started. Spring 2026 is under way.
ENROLLMENTS = []
for s in range(len(NAMES)):
    for term in range(s % 3 + 1, len(TERMS) + 1):
        for k in range(3):
            section = (term - 1) * len(COURSES) + (s + term + 3 * k) % len(COURSES) + 1
            if term < len(TERMS):
                ENROLLMENTS.append((s + 1, section, "completed", GRADES[(s * 7 + term * 5 + k * 3) % len(GRADES)]))
            else:
                ENROLLMENTS.append((s + 1, section, "enrolled", None))

build = sqlite3.connect(DATABASE)
build.executescript("""
    CREATE TABLE students (id INTEGER PRIMARY KEY, name TEXT NOT NULL, email TEXT NOT NULL UNIQUE,
                           program TEXT NOT NULL, started_on TEXT NOT NULL);
    CREATE TABLE courses (id INTEGER PRIMARY KEY, code TEXT NOT NULL UNIQUE, title TEXT NOT NULL,
                          department TEXT NOT NULL, credits INTEGER NOT NULL);
    CREATE TABLE terms (id INTEGER PRIMARY KEY, name TEXT NOT NULL UNIQUE, starts_on TEXT NOT NULL);
    CREATE TABLE sections (id INTEGER PRIMARY KEY, course_id INTEGER NOT NULL REFERENCES courses (id),
                           term_id INTEGER NOT NULL REFERENCES terms (id), capacity INTEGER NOT NULL);
    CREATE TABLE enrollments (student_id INTEGER NOT NULL REFERENCES students (id),
                              section_id INTEGER NOT NULL REFERENCES sections (id),
                              status TEXT NOT NULL, grade TEXT,
                              PRIMARY KEY (student_id, section_id));
""")
build.executemany("INSERT INTO students (name, email, program, started_on) VALUES (?, ?, ?, ?)", STUDENTS)
build.executemany("INSERT INTO courses (code, title, department, credits) VALUES (?, ?, ?, ?)", COURSES)
build.executemany("INSERT INTO terms (name, starts_on) VALUES (?, ?)", TERMS)
build.executemany("INSERT INTO sections (course_id, term_id, capacity) VALUES (?, ?, ?)", SECTIONS)
build.executemany("INSERT INTO enrollments (student_id, section_id, status, grade) VALUES (?, ?, ?, ?)", ENROLLMENTS)
build.commit()
build.close()

def college_engine(path=None, echo=False):
    """An engine for the college's database, in a file or in memory, with foreign keys enforced."""
    if path is None:                                # in memory: one connection, and one database, for every thread
        engine = create_engine("sqlite://", poolclass=StaticPool, echo=echo,
                               connect_args={"check_same_thread": False, "autocommit": False})
    else:
        engine = create_engine(f"sqlite:///{path}", echo=echo, connect_args={"autocommit": False})

    @event.listens_for(engine, "connect")
    def enforce_foreign_keys(dbapi_connection, connection_record):
        dbapi_connection.autocommit = True          # the PRAGMA does nothing inside a transaction,
        dbapi_connection.execute("PRAGMA foreign_keys = ON")
        dbapi_connection.autocommit = False         # and with autocommit=False sqlite3 keeps one open

    return engine


engine = college_engine(DATABASE)

TRANSCRIPT = text("""
    SELECT terms.name AS term, courses.code, courses.title, courses.credits, enrollments.grade
    FROM enrollments
    JOIN sections ON sections.id = enrollments.section_id
    JOIN courses ON courses.id = sections.course_id
    JOIN terms ON terms.id = sections.term_id
    WHERE enrollments.student_id = :student
    ORDER BY terms.starts_on, courses.code
""")
BY_EMAIL = text("SELECT id, name, program FROM students WHERE email = :email")
BY_PROGRAM = text("SELECT id, name FROM students WHERE program = :program ORDER BY name")
CREDITS_EARNED = text("""
    SELECT SUM(courses.credits) FROM enrollments
    JOIN sections ON sections.id = enrollments.section_id
    JOIN courses ON courses.id = sections.course_id
    WHERE enrollments.student_id = :student AND enrollments.status = 'completed' AND enrollments.grade <> 'F'
""")

GRADE_POINTS = {"A": 4.0, "A-": 3.7, "B+": 3.3, "B": 3.0, "B-": 2.7, "C+": 2.3, "C": 2.0, "C-": 1.7, "D": 1.0, "F": 0.0}


def transcript(conn, email):
    """A student's transcript as a dictionary ready for JSON, or None for an email nobody has."""
    student = conn.execute(BY_EMAIL, {"email": email}).mappings().one_or_none()
    if student is None:
        return None
    courses = conn.execute(TRANSCRIPT, {"student": student["id"]}).mappings().all()
    graded = [course for course in courses if course["grade"] is not None]
    attempted = sum(course["credits"] for course in graded)
    points = sum(GRADE_POINTS[course["grade"]] * course["credits"] for course in graded)
    return {
        "name": student["name"],
        "program": student["program"],
        "credits_attempted": attempted,
        "credits_earned": conn.execute(CREDITS_EARNED, {"student": student["id"]}).scalar_one(),
        "gpa": round(points / attempted, 2) if attempted else None,
        "courses": [dict(course) for course in courses],
    }


print("sqlalchemy", sqlalchemy.__version__, "|", DATABASE, "|", len(ENROLLMENTS), "enrollments")


sqlalchemy 2.0.54 | scratch/college.db | 228 enrollments


**1.** One student, by email.


In [2]:
with engine.connect() as conn:
    student = conn.execute(BY_EMAIL, {"email": "hali@college.edu"}).one()

print(student.name, "|", student.program)


Hassan Ali | Mathematics


An email is unique, so `one()` is the method that says so, and raises if the database ever disagrees.


**2.** One column, with `scalars()`.


In [3]:
with engine.connect() as conn:
    print(conn.execute(text("SELECT code FROM courses WHERE credits = 4 ORDER BY code")).scalars().all())


['BIO-101', 'CHE-110', 'MAT-120', 'MAT-121']


**3.** One value, with `scalar_one()`.


In [4]:
with engine.connect() as conn:
    print(conn.execute(text("SELECT COUNT(*) FROM enrollments WHERE section_id BETWEEN 31 AND 40")).scalar_one())


75


Twenty-five students taking three courses each. A `COUNT(*)` always returns exactly one row, even
when it counts nothing, so `scalar_one()` never raises here.


**4.** Dictionaries, then JSON.


In [5]:
COMPUTER_SCIENCE = text("SELECT code, title, credits FROM courses WHERE department = 'Computer Science' ORDER BY code")

with engine.connect() as conn:
    courses = conn.execute(COMPUTER_SCIENCE).mappings().all()

for course in courses:
    print(dict(course))
print(json.dumps([dict(course) for course in courses]))


{'code': 'CSC-101', 'title': 'Programming I', 'credits': 3}
{'code': 'CSC-201', 'title': 'Data Structures', 'credits': 3}
[{"code": "CSC-101", "title": "Programming I", "credits": 3}, {"code": "CSC-201", "title": "Data Structures", "credits": 3}]


**5.** Every student, ten at a time.


In [6]:
with engine.connect() as conn:
    result = conn.execute(text("SELECT id, name FROM students ORDER BY id"))
    for batch in result.partitions(10):
        print(len(batch), "rows, starting with", batch[0].name)


10 rows, starting with Ana Reyes
10 rows, starting with Keiko Tanaka
5 rows, starting with Umar Farouk


Twenty-five students make two full batches and a last one of five. A batch is a list of `Row`, so
`batch[0].name` reads the first row's column by name.


**6.** Grade point averages, highest first.


In [7]:
with engine.connect() as conn:
    emails = conn.execute(text("SELECT email FROM students WHERE started_on = '2025-08-25'")).scalars().all()
    records = [transcript(conn, email) for email in emails]

for record in sorted(records, key=lambda record: record["gpa"], reverse=True):
    print(f"{record['name']:<14} {record['gpa']:.2f}")


Felix Wagner   3.00
Isabel Costa   2.80
Rosa Delgado   2.69
Umar Farouk    2.33
Liam Murphy    2.08
Yara Haddad    1.92
Chloe Martin   1.91
Olivia Brandt  1.55


`scalars()` gave the emails as a list of strings, and `transcript` did the rest for every one of
them. The eight students who started in Fall 2025 have finished one term, so each average rests on
three courses.

Last, remove the scratch folder:


In [8]:
engine.dispose()
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


---

&#8592; **Back to:** [Reading Results](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlalchemy-deep-dive/04-reading-results.ipynb)  &nbsp;&middot;&nbsp;  [SQLAlchemy, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlalchemy-deep-dive.html)
